# 🎭 Tutoriel Playwright CLI — Tests de l'application TodoMVC

**Site testé :** https://demo.playwright.dev/todomvc  
**Outil :** `playwright-cli` + Skills Claude Code  
**Langue :** Français

---

## Comment utiliser ce notebook

Chaque scénario contient :
1. Un **prompt qualitatif** à soumettre à Claude Code (avec le skill `playwright-cli` actif)
2. Les **commandes équivalentes** à exécuter directement dans un terminal
3. Un **rapport HTML** généré automatiquement après chaque test

> **Pré-requis :** `playwright-cli install --skills` doit avoir été exécuté dans ce dossier.

---
## Scénario 1 — Ajouter des tâches

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, ouvre le navigateur sur
https://demo.playwright.dev/todomvc et ajoute exactement trois tâches :
"Acheter du lait", "Lire un livre", "Faire la vaisselle".

Après chaque ajout, prends un snapshot pour confirmer que la tâche
est bien présente dans la liste.

Vérifications attendues :
- La liste contient exactement 3 tâches
- Le compteur affiche "3 items left"
- Le champ de saisie est vide après chaque ajout
- Aucune des tâches n'est cochée (statut actif)

Génère un rapport HTML de ce test dans le fichier rapport_scenario1.html.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
# ── Ouvrir le site ──────────────────────────────────────────────────────────
playwright-cli open "https://demo.playwright.dev/todomvc"

# ── Ajouter 3 tâches ────────────────────────────────────────────────────────
playwright-cli fill e8 "Acheter du lait" --submit
playwright-cli fill e8 "Lire un livre" --submit
playwright-cli fill e8 "Faire la vaisselle" --submit

# ── Collecter les résultats ──────────────────────────────────────────────────
NB=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")
LEFT=$(playwright-cli --raw eval "document.querySelector('.todo-count strong').textContent")
VIDE=$(playwright-cli --raw eval "document.querySelector('.new-todo').value === '' ? 'oui' : 'non'")
COMP=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li.completed').length")
TACHES=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].map(l=>l.textContent).join(' | ')")

echo "Tâches trouvées : $NB"
echo "Items left      : $LEFT"
echo "Champ vide      : $VIDE"
echo "Complétées      : $COMP"
echo "Liste           : $TACHES"

# ── Statut global ────────────────────────────────────────────────────────────
if [ "$NB" = "\"3\"" ] && [ "$LEFT" = "\"3\"" ] && [ "$COMP" = "0" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

# ── Générer le rapport HTML ───────────────────────────────────────────────────
DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario1.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 1</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:860px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  tr:last-child td{border:none}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 1</h1>
  <p style="color:#666">Ajouter des tâches | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>

<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Ouvre https://demo.playwright.dev/todomvc et ajoute trois tâches :
"Acheter du lait", "Lire un livre", "Faire la vaisselle".
Vérifie que le compteur affiche 3 items left et que le champ est vide après chaque ajout.</div>
</div>

<div class="card">
  <h2>📊 Vérifications</h2>
  <table>
    <tr><th>Assertion</th><th>Attendu</th><th>Obtenu</th><th>Résultat</th></tr>
    <tr><td>Nombre de tâches dans la liste</td><td>3</td><td>${NB}</td><td class="$([ "$NB" = '"3"' ] && echo ok || echo ko)">$([ "$NB" = '"3"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Compteur items left</td><td>3</td><td>${LEFT}</td><td class="$([ "$LEFT" = '"3"' ] && echo ok || echo ko)">$([ "$LEFT" = '"3"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Champ de saisie vide</td><td>oui</td><td>${VIDE}</td><td class="$([ "$VIDE" = '"oui"' ] && echo ok || echo ko)">$([ "$VIDE" = '"oui"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Tâches complétées</td><td>0</td><td>${COMP}</td><td class="$([ "$COMP" = '0' ] && echo ok || echo ko)">$([ "$COMP" = '0' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>

<div class="card">
  <h2>📋 Tâches créées</h2>
  <p>${TACHES}</p>
</div>
</body></html>
HTMLEOF

echo "✅ Rapport généré : rapport_scenario1.html"

---
## Scénario 2 — Compléter une tâche

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli (le navigateur est déjà ouvert sur TodoMVC
avec 3 tâches), effectue les actions suivantes :

1. Coche uniquement la tâche "Acheter du lait" via sa checkbox Toggle Todo
2. Prends un snapshot pour observer le changement visuel (style barré)

Vérifications attendues :
- La tâche cochée possède la classe CSS 'completed'
- Le compteur passe à "2 items left" (pas 3)
- Les 2 autres tâches restent actives (non complétées)
- Le bouton 'Clear completed' est maintenant visible dans le footer

Génère un rapport HTML dans rapport_scenario2.html.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
# ── Cocher la première tâche ─────────────────────────────────────────────────
playwright-cli check "getByRole('listitem').filter({ hasText: 'Acheter du lait' }).getByLabel('Toggle Todo')"

# ── Collecter les résultats ──────────────────────────────────────────────────
COMP=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li.completed').length")
LEFT=$(playwright-cli --raw eval "document.querySelector('.todo-count strong').textContent")
HAS_CLASS=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li')[0].classList.contains('completed') ? 'oui' : 'non'")
ACTIVES=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li:not(.completed)').length")
BTN_CLEAR=$(playwright-cli --raw eval "document.querySelector('.clear-completed') ? 'visible' : 'absent'")

echo "Tâches complétées     : $COMP"
echo "Items left            : $LEFT"
echo "Classe 'completed'    : $HAS_CLASS"
echo "Tâches encore actives : $ACTIVES"
echo "Bouton Clear Completed: $BTN_CLEAR"

if [ "$COMP" = "1" ] && [ "$LEFT" = "\"2\"" ] && [ "$HAS_CLASS" = "\"oui\"" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario2.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 2</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:860px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 2</h1>
  <p style="color:#666">Compléter une tâche | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>
<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Coche uniquement la tâche "Acheter du lait" et vérifie que le compteur passe à 2 items left,
que la classe CSS 'completed' est appliquée, et que le bouton Clear Completed apparaît.</div>
</div>
<div class="card">
  <h2>📊 Vérifications</h2>
  <table>
    <tr><th>Assertion</th><th>Attendu</th><th>Obtenu</th><th>Résultat</th></tr>
    <tr><td>Nombre de tâches complétées</td><td>1</td><td>${COMP}</td><td class="$([ "$COMP" = '1' ] && echo ok || echo ko)">$([ "$COMP" = '1' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Compteur items left</td><td>2</td><td>${LEFT}</td><td class="$([ "$LEFT" = '"2"' ] && echo ok || echo ko)">$([ "$LEFT" = '"2"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Classe CSS 'completed' présente</td><td>oui</td><td>${HAS_CLASS}</td><td class="$([ "$HAS_CLASS" = '"oui"' ] && echo ok || echo ko)">$([ "$HAS_CLASS" = '"oui"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Tâches encore actives</td><td>2</td><td>${ACTIVES}</td><td class="$([ "$ACTIVES" = '2' ] && echo ok || echo ko)">$([ "$ACTIVES" = '2' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Bouton Clear Completed</td><td>visible</td><td>${BTN_CLEAR}</td><td class="$([ "$BTN_CLEAR" = '"visible"' ] && echo ok || echo ko)">$([ "$BTN_CLEAR" = '"visible"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>
</body></html>
HTMLEOF
echo "✅ Rapport généré : rapport_scenario2.html"

---
## Scénario 3 — Filtrer les tâches

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, teste les trois filtres de navigation
de l'application TodoMVC dans cet ordre précis :

1. Clique sur le filtre "Active"  → vérifie l'URL (#/active) et le nombre de tâches visibles
2. Clique sur le filtre "Completed" → vérifie l'URL (#/completed) et le nombre de tâches
3. Clique sur le filtre "All"     → vérifie le retour à l'état initial (#/)

Pour chaque filtre, prends un snapshot et note le nombre d'éléments li visibles.

Vérifications attendues (état courant : 3 tâches dont 1 complétée) :
- Filtre Active    : 2 tâches visibles, URL contient '#/active'
- Filtre Completed : 1 tâche visible,  URL contient '#/completed'
- Filtre All       : 3 tâches visibles, URL contient '#/'

Génère un rapport HTML dans rapport_scenario3.html avec un tableau comparatif.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
# ── Filtre Active ─────────────────────────────────────────────────────────────
playwright-cli click "getByRole('link', { name: 'Active' })"
NB_ACTIVE=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")
URL_ACTIVE=$(playwright-cli --raw eval "window.location.hash")

# ── Filtre Completed ──────────────────────────────────────────────────────────
playwright-cli click "getByRole('link', { name: 'Completed' })"
NB_COMP=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")
URL_COMP=$(playwright-cli --raw eval "window.location.hash")

# ── Filtre All ────────────────────────────────────────────────────────────────
playwright-cli click "getByRole('link', { name: 'All' })"
NB_ALL=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")
URL_ALL=$(playwright-cli --raw eval "window.location.hash")

echo "Active    : $NB_ACTIVE tâches | URL: $URL_ACTIVE"
echo "Completed : $NB_COMP tâche(s) | URL: $URL_COMP"
echo "All       : $NB_ALL tâches    | URL: $URL_ALL"

if [ "$NB_ACTIVE" = "2" ] && [ "$NB_COMP" = "1" ] && [ "$NB_ALL" = "\"3\"" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario3.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 3</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:900px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
  .filter-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-top:14px}
  .filter-box{border-radius:8px;padding:16px;text-align:center;border:2px solid #eee}
  .filter-box h3{margin:0 0 8px;font-size:1em}
  .filter-box .num{font-size:2.5em;font-weight:bold;color:#b83f45}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 3</h1>
  <p style="color:#666">Filtrage des tâches | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>
<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Teste les filtres Active, Completed et All. Vérifie le nombre de tâches visibles
et l'URL pour chaque filtre (état : 3 tâches dont 1 complétée).</div>
</div>
<div class="card">
  <h2>🔍 Résultats par filtre</h2>
  <div class="filter-grid">
    <div class="filter-box"><h3>🔵 Filtre Active</h3><div class="num">${NB_ACTIVE}</div><p>tâches visibles</p><code>${URL_ACTIVE}</code></div>
    <div class="filter-box"><h3>✅ Filtre Completed</h3><div class="num">${NB_COMP}</div><p>tâche visible</p><code>${URL_COMP}</code></div>
    <div class="filter-box"><h3>📋 Filtre All</h3><div class="num">${NB_ALL}</div><p>tâches visibles</p><code>${URL_ALL}</code></div>
  </div>
</div>
<div class="card">
  <h2>📊 Tableau de vérification</h2>
  <table>
    <tr><th>Filtre</th><th>URL attendue</th><th>Tâches attendues</th><th>Tâches obtenues</th><th>Résultat</th></tr>
    <tr><td>Active</td><td>#/active</td><td>2</td><td>${NB_ACTIVE}</td><td class="$([ "$NB_ACTIVE" = '2' ] && echo ok || echo ko)">$([ "$NB_ACTIVE" = '2' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Completed</td><td>#/completed</td><td>1</td><td>${NB_COMP}</td><td class="$([ "$NB_COMP" = '1' ] && echo ok || echo ko)">$([ "$NB_COMP" = '1' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>All</td><td>#/</td><td>3</td><td>${NB_ALL}</td><td class="$([ "$NB_ALL" = '"3"' ] && echo ok || echo ko)">$([ "$NB_ALL" = '"3"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>
</body></html>
HTMLEOF
echo "✅ Rapport généré : rapport_scenario3.html"

---
## Scénario 4 — Supprimer une tâche

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, supprime la tâche "Lire un livre"
en cliquant sur son bouton de suppression (×).

Note : ce bouton n'est visible qu'au survol. Utilise un locator ciblant
l'élément button à l'intérieur de l'item filtré par son texte.

Vérifications attendues :
- La liste passe de 3 à 2 tâches
- Le texte "Lire un livre" n'apparaît plus dans aucun élément label
- Les tâches restantes sont "Acheter du lait" et "Faire la vaisselle"
- Le compteur items left se met à jour correctement

Génère un rapport HTML dans rapport_scenario4.html.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
NB_AVANT=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")
LISTE_AVANT=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].map(l=>l.textContent).join(' | ')")

playwright-cli click "getByRole('listitem').filter({ hasText: 'Lire un livre' }).locator('button')"

NB_APRES=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")
LISTE_APRES=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].map(l=>l.textContent).join(' | ')")
ABSENT=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].some(l=>l.textContent==='Lire un livre') ? 'non' : 'oui'")
LEFT=$(playwright-cli --raw eval "document.querySelector('.todo-count strong').textContent")

echo "Avant : $NB_AVANT tâches → $LISTE_AVANT"
echo "Après : $NB_APRES tâches → $LISTE_APRES"
echo "'Lire un livre' absent : $ABSENT"
echo "Items left : $LEFT"

if [ "$NB_APRES" = "2" ] && [ "$ABSENT" = "\"oui\"" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario4.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 4</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:860px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
  .diff{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:12px}
  .diff-box{border-radius:8px;padding:14px;border:2px solid #eee}
  .deleted{text-decoration:line-through;color:#dc3545}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 4</h1>
  <p style="color:#666">Suppression d'une tâche | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>
<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Supprime la tâche "Lire un livre" via son bouton × et vérifie que la liste ne contient plus que 2 tâches.</div>
</div>
<div class="card">
  <h2>🔄 Avant / Après</h2>
  <div class="diff">
    <div class="diff-box"><h3>Avant (${NB_AVANT} tâches)</h3><p>${LISTE_AVANT}</p></div>
    <div class="diff-box"><h3>Après (${NB_APRES} tâches)</h3><p>${LISTE_APRES}</p><p class="deleted">Lire un livre</p></div>
  </div>
</div>
<div class="card">
  <h2>📊 Vérifications</h2>
  <table>
    <tr><th>Assertion</th><th>Attendu</th><th>Obtenu</th><th>Résultat</th></tr>
    <tr><td>Nombre de tâches après suppression</td><td>2</td><td>${NB_APRES}</td><td class="$([ "$NB_APRES" = '2' ] && echo ok || echo ko)">$([ "$NB_APRES" = '2' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>'Lire un livre' absent de la liste</td><td>oui</td><td>${ABSENT}</td><td class="$([ "$ABSENT" = '"oui"' ] && echo ok || echo ko)">$([ "$ABSENT" = '"oui"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Compteur items left</td><td>1</td><td>${LEFT}</td><td class="$([ "$LEFT" = '"1"' ] && echo ok || echo ko)">$([ "$LEFT" = '"1"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>
</body></html>
HTMLEOF
echo "✅ Rapport généré : rapport_scenario4.html"

---
## Scénario 5 — Modifier une tâche

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, modifie le texte de la tâche
"Faire la vaisselle" via le mécanisme d'édition en ligne de TodoMVC.

Étapes à suivre :
1. Double-cliquer sur le label de la tâche pour activer le mode édition
2. Confirmer que la classe CSS 'editing' est appliquée sur le li parent
3. Effacer le contenu actuel et saisir "Faire la vaisselle et ranger"
4. Valider par la touche Entrée (--submit)
5. Confirmer la sortie du mode édition (disparition de la classe 'editing')

Vérifications attendues :
- Avant : texte = "Faire la vaisselle"
- Après : texte = "Faire la vaisselle et ranger"
- Le nombre total de tâches reste inchangé (2)
- La classe 'editing' a disparu après validation

Génère un rapport HTML dans rapport_scenario5.html.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
TEXTE_AVANT=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].find(l=>l.textContent.includes('vaisselle'))?.textContent")
NB_AVANT=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")

playwright-cli dblclick "getByRole('listitem').filter({ hasText: 'Faire la vaisselle' }).locator('label')"
MODE_EDITION=$(playwright-cli --raw eval "document.querySelector('.todo-list li.editing') ? 'actif' : 'inactif'")

playwright-cli fill "getByRole('listitem').filter({ hasText: 'Faire la vaisselle' }).getByRole('textbox')" "Faire la vaisselle et ranger" --submit

TEXTE_APRES=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].find(l=>l.textContent.includes('vaisselle'))?.textContent")
MODE_APRES=$(playwright-cli --raw eval "document.querySelector('.todo-list li.editing') ? 'actif' : 'inactif'")
NB_APRES=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")

echo "Texte avant  : $TEXTE_AVANT"
echo "Mode édition : $MODE_EDITION"
echo "Texte après  : $TEXTE_APRES"
echo "Mode après   : $MODE_APRES"
echo "Nb tâches    : $NB_AVANT → $NB_APRES"

if [ "$TEXTE_APRES" = "\"Faire la vaisselle et ranger\"" ] && [ "$MODE_APRES" = "\"inactif\"" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario5.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 5</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:860px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
  .arrow{text-align:center;font-size:2em;color:#b83f45;padding:10px}
  .text-box{background:#f8f9fa;border-radius:6px;padding:12px 16px;font-family:monospace}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 5</h1>
  <p style="color:#666">Édition en ligne d'une tâche | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>
<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Double-cliquer sur "Faire la vaisselle", remplacer par "Faire la vaisselle et ranger",
valider avec Entrée. Vérifier la classe 'editing' avant et après.</div>
</div>
<div class="card">
  <h2>✏️ Transformation du texte</h2>
  <div class="text-box">${TEXTE_AVANT}</div>
  <div class="arrow">↓</div>
  <div class="text-box">${TEXTE_APRES}</div>
</div>
<div class="card">
  <h2>📊 Vérifications</h2>
  <table>
    <tr><th>Assertion</th><th>Attendu</th><th>Obtenu</th><th>Résultat</th></tr>
    <tr><td>Mode édition activé au double-clic</td><td>actif</td><td>${MODE_EDITION}</td><td class="$([ "$MODE_EDITION" = '"actif"' ] && echo ok || echo ko)">$([ "$MODE_EDITION" = '"actif"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Texte modifié correctement</td><td>Faire la vaisselle et ranger</td><td>${TEXTE_APRES}</td><td class="$([ "$TEXTE_APRES" = '"Faire la vaisselle et ranger"' ] && echo ok || echo ko)">$([ "$TEXTE_APRES" = '"Faire la vaisselle et ranger"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Mode édition désactivé après validation</td><td>inactif</td><td>${MODE_APRES}</td><td class="$([ "$MODE_APRES" = '"inactif"' ] && echo ok || echo ko)">$([ "$MODE_APRES" = '"inactif"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Nombre de tâches inchangé</td><td>${NB_AVANT}</td><td>${NB_APRES}</td><td class="$([ "$NB_AVANT" = "$NB_APRES" ] && echo ok || echo ko)">$([ "$NB_AVANT" = "$NB_APRES" ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>
</body></html>
HTMLEOF
echo "✅ Rapport généré : rapport_scenario5.html"

---
## Scénario 6 — Toggle All (tout cocher / tout décocher)

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, teste la fonctionnalité
"Mark all as complete" qui permet de basculer toutes les tâches d'un coup.

Étape A — Tout cocher :
- Utilise getByRole('checkbox', { name: '❯Mark all as complete' }) pour tout cocher
- Vérifie que TOUTES les tâches ont la classe 'completed'
- Vérifie que le compteur affiche "0 items left"

Étape B — Tout décocher :
- Utilise la même checkbox (uncheck) pour tout décocher
- Vérifie qu'AUCUNE tâche n'a la classe 'completed'
- Vérifie que le compteur revient au nombre total de tâches

Génère un rapport HTML dans rapport_scenario6.html avec les deux étapes.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
NB_TOTAL=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")

# ── Étape A : tout cocher ─────────────────────────────────────────────────────
playwright-cli check "getByRole('checkbox', { name: '❯Mark all as complete' })"
COMP_A=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li.completed').length")
LEFT_A=$(playwright-cli --raw eval "document.querySelector('.todo-count strong').textContent")

# ── Étape B : tout décocher ───────────────────────────────────────────────────
playwright-cli uncheck "getByRole('checkbox', { name: '❯Mark all as complete' })"
COMP_B=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li.completed').length")
LEFT_B=$(playwright-cli --raw eval "document.querySelector('.todo-count strong').textContent")

echo "Total tâches       : $NB_TOTAL"
echo "[A] Après check    : $COMP_A complétées, $LEFT_A items left"
echo "[B] Après uncheck  : $COMP_B complétées, $LEFT_B items left"

if [ "$COMP_A" = "$NB_TOTAL" ] && [ "$LEFT_A" = "\"0\"" ] && [ "$COMP_B" = "0" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario6.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 6</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:860px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
  .steps{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:14px}
  .step{border-radius:8px;padding:16px;border:2px solid #eee;text-align:center}
  .big{font-size:2.5em;font-weight:bold;color:#b83f45}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 6</h1>
  <p style="color:#666">Toggle All (tout cocher / tout décocher) | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>
<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Teste Mark All as Complete : cocher tout (0 items left) puis décocher tout (${NB_TOTAL} items left).</div>
</div>
<div class="card">
  <h2>🔄 Résultats des deux étapes</h2>
  <div class="steps">
    <div class="step"><h3>✅ Étape A — Tout cocher</h3><div class="big">${COMP_A}/${NB_TOTAL}</div><p>tâches complétées</p><p><strong>${LEFT_A}</strong> items left</p></div>
    <div class="step"><h3>☐ Étape B — Tout décocher</h3><div class="big">${COMP_B}/${NB_TOTAL}</div><p>tâches complétées</p><p><strong>${LEFT_B}</strong> items left</p></div>
  </div>
</div>
<div class="card">
  <h2>📊 Vérifications</h2>
  <table>
    <tr><th>Assertion</th><th>Attendu</th><th>Obtenu</th><th>Résultat</th></tr>
    <tr><td>[A] Toutes les tâches complétées</td><td>${NB_TOTAL}</td><td>${COMP_A}</td><td class="$([ "$COMP_A" = "$NB_TOTAL" ] && echo ok || echo ko)">$([ "$COMP_A" = "$NB_TOTAL" ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>[A] Items left = 0</td><td>0</td><td>${LEFT_A}</td><td class="$([ "$LEFT_A" = '"0"' ] && echo ok || echo ko)">$([ "$LEFT_A" = '"0"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>[B] Aucune tâche complétée</td><td>0</td><td>${COMP_B}</td><td class="$([ "$COMP_B" = '0' ] && echo ok || echo ko)">$([ "$COMP_B" = '0' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>[B] Items left = ${NB_TOTAL}</td><td>${NB_TOTAL}</td><td>${LEFT_B}</td><td class="$([ "$LEFT_B" = "\"$NB_TOTAL\"" ] && echo ok || echo ko)">$([ "$LEFT_B" = "\"$NB_TOTAL\"" ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>
</body></html>
HTMLEOF
echo "✅ Rapport généré : rapport_scenario6.html"

---
## Scénario 7 — Supprimer les tâches complétées (Clear Completed)

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, teste la suppression en lot
des tâches complétées via le bouton "Clear completed".

Étapes :
1. Utilise "Mark all as complete" pour cocher toutes les tâches d'un coup
2. Vérifie que le bouton "Clear completed" est bien visible dans le footer
3. Clique sur ce bouton
4. Prends un snapshot pour constater l'état final

Vérifications attendues :
- La liste est vide (0 tâches)
- Le footer disparaît (plus de tâches à afficher)
- Le bouton "Clear completed" n'est plus visible
- La section .main (toggle-all + liste) est masquée

Génère un rapport HTML dans rapport_scenario7.html.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
NB_AVANT=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")

playwright-cli check "getByRole('checkbox', { name: '❯Mark all as complete' })"
BTN_AVANT=$(playwright-cli --raw eval "document.querySelector('.clear-completed') ? 'visible' : 'absent'")

playwright-cli click "getByRole('button', { name: 'Clear completed' })"

NB_APRES=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")
BTN_APRES=$(playwright-cli --raw eval "document.querySelector('.clear-completed') ? 'visible' : 'absent'")
FOOTER=$(playwright-cli --raw eval "document.querySelector('.footer') ? 'visible' : 'masqué'")
MAIN=$(playwright-cli --raw eval "document.querySelector('.main') ? 'visible' : 'masqué'")

echo "Avant clear  : $NB_AVANT tâches"
echo "Bouton avant : $BTN_AVANT"
echo "Après clear  : $NB_APRES tâches"
echo "Bouton après : $BTN_APRES"
echo "Footer       : $FOOTER"
echo "Section main : $MAIN"

if [ "$NB_APRES" = "0" ] && [ "$BTN_APRES" = "\"absent\"" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario7.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 7</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:860px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
  .stat{display:inline-block;background:#b83f45;color:#fff;border-radius:8px;padding:10px 20px;margin:6px;font-size:1.3em;font-weight:bold}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 7</h1>
  <p style="color:#666">Clear Completed | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>
<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Cocher tout, cliquer sur Clear Completed, vérifier que la liste est vide et que le footer disparaît.</div>
</div>
<div class="card">
  <h2>📈 Évolution de la liste</h2>
  <span class="stat">Avant : ${NB_AVANT} tâches</span>
  <span style="font-size:1.5em"> → </span>
  <span class="stat">Après : ${NB_APRES} tâche</span>
</div>
<div class="card">
  <h2>📊 Vérifications</h2>
  <table>
    <tr><th>Assertion</th><th>Attendu</th><th>Obtenu</th><th>Résultat</th></tr>
    <tr><td>Bouton Clear visible avant action</td><td>visible</td><td>${BTN_AVANT}</td><td class="$([ "$BTN_AVANT" = '"visible"' ] && echo ok || echo ko)">$([ "$BTN_AVANT" = '"visible"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Liste vide après Clear</td><td>0</td><td>${NB_APRES}</td><td class="$([ "$NB_APRES" = '0' ] && echo ok || echo ko)">$([ "$NB_APRES" = '0' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Bouton Clear absent après</td><td>absent</td><td>${BTN_APRES}</td><td class="$([ "$BTN_APRES" = '"absent"' ] && echo ok || echo ko)">$([ "$BTN_APRES" = '"absent"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Footer masqué</td><td>masqué</td><td>${FOOTER}</td><td class="$([ "$FOOTER" = '"masqué"' ] && echo ok || echo ko)">$([ "$FOOTER" = '"masqué"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>
</body></html>
HTMLEOF
echo "✅ Rapport généré : rapport_scenario7.html"

---
## Scénario 8 — Annuler une édition avec Escape

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, vérifie que la touche Escape
annule une édition en cours sans persister les modifications.

Étapes :
1. Ajoute une nouvelle tâche "Tâche originale avant Escape"
2. Double-clique sur son label pour entrer en mode édition
3. Remplace le texte par "Ce texte ne doit PAS être sauvegardé"
4. Appuie sur Escape SANS appuyer sur Entrée
5. Prends un snapshot

Vérifications attendues :
- La classe 'editing' a bien disparu du li après Escape
- Le label affiche toujours "Tâche originale avant Escape" (texte non modifié)
- Le texte "Ce texte ne doit PAS être sauvegardé" est absent de la liste
- Le nombre de tâches est inchangé

Génère un rapport HTML dans rapport_scenario8.html.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
playwright-cli fill e8 "Tâche originale avant Escape" --submit
NB=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")

playwright-cli dblclick "getByRole('listitem').filter({ hasText: 'Tâche originale avant Escape' }).locator('label')"
MODE_AVANT=$(playwright-cli --raw eval "document.querySelector('.todo-list li.editing') ? 'actif' : 'inactif'")

playwright-cli fill "getByRole('listitem').filter({ hasText: 'Tâche originale avant Escape' }).getByRole('textbox')" "Ce texte ne doit PAS être sauvegardé"
playwright-cli press Escape

MODE_APRES=$(playwright-cli --raw eval "document.querySelector('.todo-list li.editing') ? 'actif' : 'inactif'")
TEXTE=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].find(l=>l.textContent.includes('Escape'))?.textContent")
MODIFIE=$(playwright-cli --raw eval "[...document.querySelectorAll('.todo-list li label')].some(l=>l.textContent.includes('PAS')) ? 'présent' : 'absent'")
NB_APRES=$(playwright-cli --raw eval "document.querySelectorAll('.todo-list li').length")

echo "Mode édition avant Escape : $MODE_AVANT"
echo "Mode édition après Escape : $MODE_APRES"
echo "Texte de la tâche         : $TEXTE"
echo "Texte annulé (absent ?)   : $MODIFIE"
echo "Nb tâches inchangé        : $NB → $NB_APRES"

if [ "$MODE_APRES" = "\"inactif\"" ] && [ "$TEXTE" = "\"Tâche originale avant Escape\"" ] && [ "$MODIFIE" = "\"absent\"" ]; then
  STATUT="RÉUSSI"; COULEUR="#28a745"
else
  STATUT="ÉCHOUÉ"; COULEUR="#dc3545"
fi

DATE=$(date '+%d/%m/%Y à %H:%M:%S')
cat > rapport_scenario8.html << HTMLEOF
<!DOCTYPE html>
<html lang="fr">
<head><meta charset="UTF-8"><title>Rapport — Scénario 8</title>
<style>
  body{font-family:Segoe UI,Arial,sans-serif;max-width:860px;margin:40px auto;padding:20px;background:#f5f5f5}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.08)}
  h1{color:#b83f45;margin:0 0 4px}
  .badge{display:inline-block;padding:6px 18px;border-radius:20px;color:#fff;font-weight:bold;font-size:1.1em;background:${COULEUR}}
  table{width:100%;border-collapse:collapse;margin-top:12px}
  th{background:#b83f45;color:#fff;padding:10px 14px;text-align:left}
  td{padding:9px 14px;border-bottom:1px solid #eee}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .prompt{background:#f0f4ff;border-left:4px solid #4a6cf7;padding:14px 18px;border-radius:0 8px 8px 0;font-style:italic;white-space:pre-wrap}
  .kbd{display:inline-block;background:#eee;border:1px solid #999;border-radius:4px;padding:2px 8px;font-family:monospace;font-size:1.1em}
  .timeline{border-left:3px solid #b83f45;margin:14px 0;padding-left:16px}
  .tl-step{margin:10px 0;padding:8px 12px;background:#f8f9fa;border-radius:4px}
</style></head>
<body>
<div class="card">
  <h1>🎭 Rapport de test — Scénario 8</h1>
  <p style="color:#666">Annulation d'édition par <span class="kbd">Escape</span> | ${DATE}</p>
  <span class="badge">${STATUT}</span>
</div>
<div class="card">
  <h2>📝 Prompt soumis</h2>
  <div class="prompt">Double-cliquer, saisir un texte de remplacement, appuyer sur Escape.
Vérifier que le texte original est conservé et que le mode édition est fermé.</div>
</div>
<div class="card">
  <h2>⏱️ Chronologie du test</h2>
  <div class="timeline">
    <div class="tl-step">1. Tâche créée : <strong>"Tâche originale avant Escape"</strong></div>
    <div class="tl-step">2. Double-clic → mode édition : <strong>${MODE_AVANT}</strong></div>
    <div class="tl-step">3. Saisie de "Ce texte ne doit PAS être sauvegardé"</div>
    <div class="tl-step">4. Touche <span class="kbd">Escape</span> → mode édition : <strong>${MODE_APRES}</strong></div>
    <div class="tl-step">5. Texte final dans la liste : <strong>${TEXTE}</strong></div>
  </div>
</div>
<div class="card">
  <h2>📊 Vérifications</h2>
  <table>
    <tr><th>Assertion</th><th>Attendu</th><th>Obtenu</th><th>Résultat</th></tr>
    <tr><td>Mode édition activé au double-clic</td><td>actif</td><td>${MODE_AVANT}</td><td class="$([ "$MODE_AVANT" = '"actif"' ] && echo ok || echo ko)">$([ "$MODE_AVANT" = '"actif"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Mode édition fermé après Escape</td><td>inactif</td><td>${MODE_APRES}</td><td class="$([ "$MODE_APRES" = '"inactif"' ] && echo ok || echo ko)">$([ "$MODE_APRES" = '"inactif"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Texte original conservé</td><td>Tâche originale avant Escape</td><td>${TEXTE}</td><td class="$([ "$TEXTE" = '"Tâche originale avant Escape"' ] && echo ok || echo ko)">$([ "$TEXTE" = '"Tâche originale avant Escape"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Texte annulé absent de la liste</td><td>absent</td><td>${MODIFIE}</td><td class="$([ "$MODIFIE" = '"absent"' ] && echo ok || echo ko)">$([ "$MODIFIE" = '"absent"' ] && echo '✅ OK' || echo '❌ KO')</td></tr>
    <tr><td>Nombre de tâches inchangé</td><td>${NB}</td><td>${NB_APRES}</td><td class="$([ "$NB" = "$NB_APRES" ] && echo ok || echo ko)">$([ "$NB" = "$NB_APRES" ] && echo '✅ OK' || echo '❌ KO')</td></tr>
  </table>
</div>
</body></html>
HTMLEOF
echo "✅ Rapport généré : rapport_scenario8.html"

---
## Rapport de synthèse — Tableau de bord global

### 📝 Prompt qualitatif

```
En utilisant le skill playwright-cli, génère un rapport HTML de synthèse
qui agrège les résultats des 8 scénarios de test exécutés sur TodoMVC.

Le rapport doit inclure :
- Un en-tête avec le nom de la suite, la date et un badge de statut global
- Un tableau récapitulatif de tous les scénarios (nom, statut, description)
- Des métriques visuelles : total tests, réussis, échoués, taux de succès
- Une section "Couverture fonctionnelle" listant les fonctionnalités testées

Sauvegarde dans rapport_synthese.html.
```

### ▶️ Commandes playwright-cli

In [ ]:
%%bash
playwright-cli close 2>/dev/null || true

DATE=$(date '+%d/%m/%Y à %H:%M:%S')

cat > rapport_synthese.html << 'HTMLEOF'
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>Rapport de synthèse — TodoMVC</title>
<style>
  *{box-sizing:border-box}
  body{font-family:Segoe UI,Arial,sans-serif;max-width:960px;margin:0 auto;padding:30px 20px;background:#f0f2f5}
  .header{background:linear-gradient(135deg,#b83f45,#c0392b);color:#fff;border-radius:12px;padding:30px 32px;margin-bottom:20px}
  .header h1{margin:0 0 6px;font-size:1.8em}
  .header p{margin:0;opacity:.85}
  .badge{display:inline-block;padding:5px 16px;border-radius:20px;font-weight:bold;font-size:.95em}
  .badge-ok{background:#28a745;color:#fff}
  .badge-ko{background:#dc3545;color:#fff}
  .card{background:#fff;border-radius:10px;padding:24px;margin:16px 0;box-shadow:0 2px 8px rgba(0,0,0,.07)}
  .metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:16px 0}
  .metric{background:#fff;border-radius:10px;padding:18px;text-align:center;box-shadow:0 2px 8px rgba(0,0,0,.07)}
  .metric .num{font-size:2.4em;font-weight:bold;line-height:1}
  .metric .lbl{font-size:.85em;color:#666;margin-top:4px}
  .m-blue .num{color:#4a6cf7}
  .m-green .num{color:#28a745}
  .m-red .num{color:#dc3545}
  .m-orange .num{color:#fd7e14}
  table{width:100%;border-collapse:collapse}
  th{background:#b83f45;color:#fff;padding:11px 14px;text-align:left;font-weight:600}
  td{padding:10px 14px;border-bottom:1px solid #f0f0f0}
  tr:hover td{background:#fafafa}
  .ok{color:#28a745;font-weight:bold} .ko{color:#dc3545;font-weight:bold}
  .num-circle{display:inline-block;width:26px;height:26px;border-radius:50%;background:#b83f45;color:#fff;text-align:center;line-height:26px;font-size:.85em;font-weight:bold;margin-right:6px}
  .coverage{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-top:14px}
  .cov-item{background:#f8f9fa;border-radius:8px;padding:12px 14px;border-left:4px solid #b83f45}
  .cov-item h4{margin:0 0 4px;color:#b83f45}
  .cov-item p{margin:0;font-size:.9em;color:#555}
  .progress-bar{background:#eee;border-radius:20px;height:20px;margin-top:12px;overflow:hidden}
  .progress-fill{background:linear-gradient(90deg,#28a745,#20c997);height:100%;border-radius:20px;display:flex;align-items:center;justify-content:center;color:#fff;font-weight:bold;font-size:.85em;transition:width .5s}
</style>
</head>
<body>

<div class="header">
  <h1>🎭 Rapport de Synthèse — TodoMVC</h1>
  <p>Suite de tests Playwright CLI · https://demo.playwright.dev/todomvc</p>
  <p style="margin-top:8px">Date d'exécution : <strong>DATEPLACEHOLDER</strong></p>
</div>

<div class="metrics">
  <div class="metric m-blue"><div class="num">8</div><div class="lbl">Scénarios exécutés</div></div>
  <div class="metric m-green"><div class="num" id="ok-count">—</div><div class="lbl">Réussis</div></div>
  <div class="metric m-red"><div class="num" id="ko-count">—</div><div class="lbl">Échoués</div></div>
  <div class="metric m-orange"><div class="num" id="taux">—</div><div class="lbl">Taux de succès</div></div>
</div>

<div class="card">
  <h2>📊 Récapitulatif des scénarios</h2>
  <table id="results-table">
    <tr><th>#</th><th>Scénario</th><th>Fonctionnalité testée</th><th>Rapport</th><th>Statut</th></tr>
    <tr><td><span class="num-circle">1</span></td><td>Ajouter des tâches</td><td>Saisie + Enter, compteur items left</td><td><a href="rapport_scenario1.html">📄 Voir</a></td><td class="ok" id="s1">⏳</td></tr>
    <tr><td><span class="num-circle">2</span></td><td>Compléter une tâche</td><td>Checkbox toggle, classe CSS completed</td><td><a href="rapport_scenario2.html">📄 Voir</a></td><td class="ok" id="s2">⏳</td></tr>
    <tr><td><span class="num-circle">3</span></td><td>Filtrer les tâches</td><td>Filtres All / Active / Completed</td><td><a href="rapport_scenario3.html">📄 Voir</a></td><td class="ok" id="s3">⏳</td></tr>
    <tr><td><span class="num-circle">4</span></td><td>Supprimer une tâche</td><td>Bouton × au survol</td><td><a href="rapport_scenario4.html">📄 Voir</a></td><td class="ok" id="s4">⏳</td></tr>
    <tr><td><span class="num-circle">5</span></td><td>Modifier une tâche</td><td>Double-clic, mode editing, validation</td><td><a href="rapport_scenario5.html">📄 Voir</a></td><td class="ok" id="s5">⏳</td></tr>
    <tr><td><span class="num-circle">6</span></td><td>Toggle All</td><td>Mark all / Unmark all</td><td><a href="rapport_scenario6.html">📄 Voir</a></td><td class="ok" id="s6">⏳</td></tr>
    <tr><td><span class="num-circle">7</span></td><td>Clear Completed</td><td>Suppression en lot, footer masqué</td><td><a href="rapport_scenario7.html">📄 Voir</a></td><td class="ok" id="s7">⏳</td></tr>
    <tr><td><span class="num-circle">8</span></td><td>Annuler avec Escape</td><td>Annulation édition, texte conservé</td><td><a href="rapport_scenario8.html">📄 Voir</a></td><td class="ok" id="s8">⏳</td></tr>
  </table>
  <div class="progress-bar"><div class="progress-fill" style="width:100%" id="progress">Calcul...</div></div>
</div>

<div class="card">
  <h2>🗺️ Couverture fonctionnelle</h2>
  <div class="coverage">
    <div class="cov-item"><h4>✏️ Création</h4><p>Ajout via le champ texte + touche Entrée. Vide le champ après validation.</p></div>
    <div class="cov-item"><h4>✅ Complétion</h4><p>Checkbox individuelle et Toggle All pour basculer l'état completed.</p></div>
    <div class="cov-item"><h4>🔍 Navigation</h4><p>Filtres All / Active / Completed avec routage hashbang (#/active etc.).</p></div>
    <div class="cov-item"><h4>🗑️ Suppression</h4><p>Bouton × par tâche et Clear Completed pour suppression en lot.</p></div>
    <div class="cov-item"><h4>✏️ Édition</h4><p>Double-clic sur le label, validation Entrée, annulation Escape.</p></div>
    <div class="cov-item"><h4>📊 Compteur</h4><p>items left mis à jour dynamiquement selon l'état de chaque tâche.</p></div>
  </div>
</div>

<script>
// Lire les rapports individuels via fetch pour afficher les statuts réels
async function loadStatus(id, file) {
  try {
    const r = await fetch(file);
    const text = await r.text();
    const match = text.match(/background:(#[0-9a-f]+)/ );
    const ok = text.includes('28a745');
    const el = document.getElementById(id);
    el.textContent = ok ? '✅ RÉUSSI' : '❌ ÉCHOUÉ';
    el.className = ok ? 'ok' : 'ko';
    return ok;
  } catch(e) { return null; }
}
async function init() {
  const results = await Promise.all([
    loadStatus('s1','rapport_scenario1.html'),
    loadStatus('s2','rapport_scenario2.html'),
    loadStatus('s3','rapport_scenario3.html'),
    loadStatus('s4','rapport_scenario4.html'),
    loadStatus('s5','rapport_scenario5.html'),
    loadStatus('s6','rapport_scenario6.html'),
    loadStatus('s7','rapport_scenario7.html'),
    loadStatus('s8','rapport_scenario8.html'),
  ]);
  const valid = results.filter(r => r !== null);
  const ok = valid.filter(Boolean).length;
  const total = valid.length;
  const taux = total > 0 ? Math.round(ok/total*100) : 0;
  document.getElementById('ok-count').textContent = ok;
  document.getElementById('ko-count').textContent = total - ok;
  document.getElementById('taux').textContent = taux + '%';
  const bar = document.getElementById('progress');
  bar.style.width = taux + '%';
  bar.textContent = taux + '% de succès';
  bar.style.background = taux === 100 ? 'linear-gradient(90deg,#28a745,#20c997)'
    : taux >= 75 ? 'linear-gradient(90deg,#fd7e14,#ffc107)'
    : 'linear-gradient(90deg,#dc3545,#e74c3c)';
}
init();
</script>
</body></html>
HTMLEOF

# Injecter la date réelle
DATE=$(date '+%d/%m/%Y à %H:%M:%S')
sed -i "s/DATEPLACEHOLDER/${DATE}/" rapport_synthese.html

echo "✅ Rapport de synthèse généré : rapport_synthese.html"
echo ""
echo "📂 Fichiers générés :"
ls -1 rapport_*.html

---
## Référence rapide — Sélecteurs TodoMVC

| Élément | Locator Playwright recommandé |
|---------|------------------------------|
| Champ de saisie | `getByRole('textbox', { name: 'What needs to be done?' })` |
| Toggle All | `getByRole('checkbox', { name: '❯Mark all as complete' })` |
| Checkbox d'une tâche | `getByRole('listitem').filter({ hasText: 'X' }).getByLabel('Toggle Todo')` |
| Label d'une tâche | `getByRole('listitem').filter({ hasText: 'X' }).locator('label')` |
| Champ d'édition | `getByRole('listitem').filter({ hasText: 'X' }).getByRole('textbox')` |
| Bouton × | `getByRole('listitem').filter({ hasText: 'X' }).locator('button')` |
| Filtre All | `getByRole('link', { name: 'All' })` |
| Filtre Active | `getByRole('link', { name: 'Active' })` |
| Filtre Completed | `getByRole('link', { name: 'Completed' })` |
| Clear Completed | `getByRole('button', { name: 'Clear completed' })` |

| Assertion JS | Ce que ça retourne |
|-------------|-------------------|
| `document.querySelectorAll('.todo-list li').length` | Nombre total de tâches |
| `document.querySelectorAll('.todo-list li.completed').length` | Tâches complétées |
| `document.querySelector('.todo-count strong').textContent` | Valeur du compteur |
| `document.querySelector('.todo-list li.editing')` | null si pas en édition |
| `document.querySelector('.clear-completed')` | null si bouton absent |
| `window.location.hash` | Filtre actif (#/, #/active, #/completed) |